In [ ]:

# %% Cell 1 — Setup
import subprocess, sys, os, time, math, json, shutil
import dataclasses
from dataclasses import dataclass, asdict, field
from typing import Optional, Dict, Any, List, Tuple
from contextlib import contextmanager, nullcontext
from pathlib import Path

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
install("torch")
install("numpy")
install("tqdm")
install("requests")

import torch
import torch.nn.functional as F
from torch import nn, Tensor
import numpy as np
from tqdm import tqdm

print(f"PyTorch {torch.__version__}")
assert torch.cuda.is_available(), "Need GPU!"
GPU = torch.cuda.get_device_name(0)
VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {GPU} ({VRAM:.0f}GB)")

DEVICE = "cuda"
DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
print(f"Precision: {DTYPE}")


# %% Cell 2 — Model Architecture
@dataclass
class BDHConfig:
    n_layer: int = 6
    n_embd: int = 192          # D
    n_head: int = 4            # H
    dropout: float = 0.1
    vocab_size: int = 256      # UTF-8 bytes
    mlp_dim_mult: int = 64     # N = D * this / H per head

    @property
    def N(self) -> int:
        """Neurons per head."""
        return self.n_embd * self.mlp_dim_mult // self.n_head

    @property
    def N_total(self) -> int:
        """Total neurons across all heads."""
        return self.N * self.n_head


class BDH(nn.Module):

    def __init__(self, config: BDHConfig):
        super().__init__()
        self.config = config
        C = config
        D, H, N = C.n_embd, C.n_head, C.N

        # Embedding + positional encoding
        self.embed = nn.Embedding(C.vocab_size, D)
        nn.init.normal_(self.embed.weight, std=0.02)

        # Learnable positional embedding (simpler than sinusoidal, works well)
        self.pos_emb = nn.Embedding(4096, D)  # Up to 4096 positions
        nn.init.normal_(self.pos_emb.weight, std=0.01)

        # LayerNorm (no learnable params — paper uses this)
        self.ln = nn.LayerNorm(D, elementwise_affine=False, bias=False)

        # Paper: E (encoder N→D), Dₓ (decoder D→N), Dᵧ (decoder D→N)
        self.encoder = nn.Parameter(torch.empty(H * N, D).normal_(std=1/math.sqrt(H * N)))
        self.decoder_x = nn.Parameter(torch.empty(H, D, N).normal_(std=1/math.sqrt(D)))
        self.decoder_y = nn.Parameter(torch.empty(H, D, N).normal_(std=1/math.sqrt(D)))

        # Output head
        self.lm_head = nn.Parameter(torch.empty(D, C.vocab_size).normal_(std=1/math.sqrt(D)))

        self.drop = nn.Dropout(C.dropout)

        # RoPE frequencies for attention
        freqs = self._build_rope_freqs(N)
        self.register_buffer('rope_freqs', freqs)  # (1, 1, 1, N)

        # Extraction mode
        self._extract = False
        self._buffer: Dict[int, Dict[str, Tensor]] = {}

    def _build_rope_freqs(self, N, theta=2**16):
        """Build RoPE frequencies for sparse attention."""
        idx = torch.arange(N, dtype=torch.float32)
        # Quantize to pairs (paper does this)
        idx_q = (idx / 2).floor() * 2
        freqs = 1.0 / (theta ** (idx_q / N)) / (2 * math.pi)
        return freqs.view(1, 1, 1, N)

    def _causal_attention(self, Q, K, V):
        """Parallel causal attention — single matmul, no loops.

        Q, K: (B, H, T, N) — sparse after ReLU
        V:    (B, H, T, D) — dense state vectors

        Returns: (B, H, T, D)

        This is mathematically equivalent to the recurrent ρ form
        for within-sequence attention (paper Eq. 17), but computed
        as a single batched matmul
        """
        B, H, T, N = Q.size()

        # Apply RoPE to make attention position-aware
        positions = torch.arange(T, device=Q.device, dtype=Q.dtype).view(1, 1, T, 1)
        phases = (positions * self.rope_freqs[:,:,:,:N].to(Q.dtype) % 1.0) * (2 * math.pi)

        # Rotate Q (apply RoPE)
        cos_p = torch.cos(phases)
        sin_p = torch.sin(phases)
        # Simple rotation: interleave pairs
        Q_rot = torch.stack((-Q[..., 1::2], Q[..., ::2]), dim=-1).reshape_as(Q)
        Q_roped = Q * cos_p + Q_rot * sin_p

        # Compute attention scores: (B, H, T, T)
        scores = torch.matmul(Q_roped, Q_roped.transpose(-2, -1))

        # Causal mask: only attend to past (strict causal, not including self)
        causal_mask = torch.tril(torch.ones(T, T, device=Q.device, dtype=torch.bool), diagonal=-1)
        scores = scores.masked_fill(~causal_mask, 0.0)

        # Apply attention to values
        output = torch.matmul(scores, V)  # (B, H, T, D)

        return output, scores

    @contextmanager
    def extracting(self):
        """Enable extraction of intermediate activations."""
        self._extract = True
        self._buffer = {}
        try:
            yield self._buffer
        finally:
            self._extract = False
            self._buffer = {}

    def forward(self, idx, targets=None):
        """Forward pass — fully parallel, no Python loops over T.

        Args:
            idx: (B, T) input token indices
            targets: (B, T) target token indices for loss

        Returns:
            logits: (B, T, vocab_size)
            loss: scalar or None
        """
        C = self.config
        B, T = idx.size()
        D, H, N = C.n_embd, C.n_head, C.N

        # Embed + positional
        positions = torch.arange(T, device=idx.device)
        v_ast = self.embed(idx) + self.pos_emb(positions)  # (B, T, D)
        v_ast = v_ast.unsqueeze(1)  # (B, 1, T, D) — add head dim for broadcast

        for L in range(C.n_layer):
            v_normed = self.ln(v_ast.squeeze(1)).unsqueeze(1)  # (B, 1, T, D)

            # ═══ Step 1: x = ReLU(v* @ Dₓ) ═══
            # decoder_x: (H, D, N) — project from D to N per head
            # v_normed: (B, 1, T, D) — broadcasts across H
            x_pre = torch.einsum('bitd,hdn->bhtn', v_normed, self.decoder_x)  # (B,H,T,N)
            x = F.relu(x_pre)  # Sparsity emerges naturally from ReLU

            # ═══ Step 2: a* = CausalAttn(Q=x, K=x, V=v*) ═══
            V_heads = v_normed.expand(B, H, T, D)
            a_ast, attn_scores = self._causal_attention(Q=x, K=x, V=V_heads)

            # ═══ Step 3: y = ReLU(a* @ Dᵧ) ⊙ x ═══
            y_pre = torch.einsum('bhtd,hdn->bhtn', a_ast, self.decoder_y)  # (B,H,T,N)
            y = F.relu(y_pre)
            y = y * x  # Gating: only co-active neurons survive

            # ═══ Extraction ═══
            if self._extract:
                self._buffer[L] = {
                    'x_pre': x_pre.detach().cpu(),
                    'x': x.detach().cpu(),
                    'y_pre': y_pre.detach().cpu(),
                    'y': y.detach().cpu(),
                    'a_ast': a_ast.detach().cpu(),
                    'v_ast': v_normed.detach().cpu(),
                    'attn_scores': attn_scores.detach().cpu(),
                }

            # ═══ Step 4: v* += y @ E ═══
            y_flat = y.permute(0, 2, 1, 3).reshape(B, T, H * N)  # (B, T, N_total)
            y_flat = self.drop(y_flat)
            delta = torch.matmul(y_flat, self.encoder)  # (B, T, D)
            v_ast = v_ast + delta.unsqueeze(1)

        # Output
        logits = torch.matmul(v_ast.squeeze(1), self.lm_head)  # (B, T, vocab)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, C.vocab_size), targets.view(-1))

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=5):
        """Autoregressive generation."""
        for _ in range(max_new_tokens):
            # Crop to max position embedding length
            idx_crop = idx[:, -4096:]
            logits, _ = self(idx_crop)
            logits = logits[:, -1, :] / temperature
            if torch.isnan(logits).any():
                idx_next = torch.randint(0, logits.size(-1), (idx.size(0), 1), device=idx.device)
            else:
                if top_k:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = float('-inf')
                probs = F.softmax(logits, dim=-1).clamp(min=1e-8)
                probs /= probs.sum(dim=-1, keepdim=True)
                idx_next = torch.multinomial(probs, 1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


def compute_sparsity(x):
    """Fraction of zeros."""
    return 1.0 - ((x > 0).sum().item() / max(x.numel(), 1))


# Quick sanity check
_test_cfg = BDHConfig()
_test_model = BDH(_test_cfg).to(DEVICE)
_test_input = torch.randint(0, 256, (2, 64), device=DEVICE)
with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
    _logits, _loss = _test_model(_test_input, _test_input)
print(f" Model sanity check passed. Output shape: {_logits.shape}, Loss: {_loss.item():.3f}")
n_params = sum(p.numel() for p in _test_model.parameters())
print(f"   Params: {n_params:,} ({n_params/1e6:.1f}M)")
print(f"   Config: {_test_cfg.n_layer}L/{_test_cfg.n_embd}D/{_test_cfg.n_head}H, N={_test_cfg.N}/head, N_total={_test_cfg.N_total}")
del _test_model, _test_input, _logits, _loss
torch.cuda.empty_cache()


# %% Cell 3 — Download Europarl Data
import tarfile, requests

def download_europarl(lang_pair, data_dir):
    """Download Europarl, interleave as BDH translation format, save as .bin"""
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)

    if (data_dir / "train.bin").exists():
        sz = (data_dir / "train.bin").stat().st_size
        print(f"  {data_dir}/train.bin exists ({sz/1e6:.1f}MB), skipping.")
        return

    urls = {
        "en-fr": "https://www.statmt.org/europarl/v7/fr-en.tgz",
        "en-pt": "https://www.statmt.org/europarl/v7/pt-en.tgz",
    }
    tgt_lang = lang_pair.split("-")[1]
    archive = data_dir / "europarl.tgz"

    if not archive.exists():
        print(f"   Downloading {lang_pair}...")
        resp = requests.get(urls[lang_pair], stream=True, timeout=120)
        resp.raise_for_status()
        total = int(resp.headers.get('content-length', 0))
        with open(archive, 'wb') as f:
            for chunk in tqdm(resp.iter_content(8192), total=total//8192, desc="  DL"):
                f.write(chunk)

    print(f"  Extracting...")
    with tarfile.open(archive, 'r:gz') as tar:
        tar.extractall(data_dir)

    en_file = list(data_dir.rglob("*.en"))[0]
    tgt_file = list(data_dir.rglob(f"*.{tgt_lang}"))[0]

    en_lines = open(en_file, encoding='latin-1').readlines()
    tgt_lines = open(tgt_file, encoding='latin-1').readlines()
    n = min(len(en_lines), len(tgt_lines))
    print(f"  Pairs: {n:,}")

    raw = bytearray()
    for i in tqdm(range(n), desc="  Building"):
        en = en_lines[i].strip()
        tgt = tgt_lines[i].strip()
        if len(en) < 5 or len(tgt) < 5:
            continue
        raw.extend(f"<F:en>{en}<T:{tgt_lang}>{tgt}".encode('utf-8', errors='replace'))

    split = int(len(raw) * 0.95)
    for name, chunk in [("train.bin", raw[:split]), ("val.bin", raw[split:])]:
        np.frombuffer(bytes(chunk), dtype=np.uint8).tofile(str(data_dir / name))
        print(f"  {name}: {len(chunk)/1e6:.1f}MB")

print(" French data:")
download_europarl("en-fr", "data/en-fr")
print("\n Portuguese data:")
download_europarl("en-pt", "data/en-pt")


# %% Cell 4 — Dataset + Training Config

class ByteDataset:
    def __init__(self, path, block_size):
        self.data = np.memmap(path, dtype=np.uint8, mode='r')
        self.block_size = block_size
        print(f"  {path}: {len(self.data)/1e6:.1f}MB")

    def __len__(self):
        return len(self.data) - self.block_size - 1

    def get_batch(self, batch_size, device):
        ix = torch.randint(len(self), (batch_size,))
        x = torch.stack([torch.from_numpy(self.data[i:i+self.block_size].astype(np.int64)) for i in ix])
        y = torch.stack([torch.from_numpy(self.data[i+1:i+1+self.block_size].astype(np.int64)) for i in ix])
        return x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)


@dataclass
class TrainConfig:
    # Data
    train_data: str = "data/en-fr/train.bin"
    val_data: str = "data/en-fr/val.bin"
    # Architecture
    n_layer: int = 6
    n_embd: int = 192
    n_head: int = 4
    mlp_dim_mult: int = 64    # N=3072/head, N_total=12288
    dropout: float = 0.1
    vocab_size: int = 256
    # Training — tuned for A100
    batch_size: int = 32       # A100 can handle this easily now
    block_size: int = 512      # Longer context for better translation
    max_iters: int = 50000
    learning_rate: float = 1e-3
    min_lr: float = 1e-4
    warmup_iters: int = 1000
    weight_decay: float = 0.1
    grad_clip: float = 1.0
    gradient_accumulation_steps: int = 2  # Effective batch = 64 × 512 = 32K tok/step
    # Logging
    log_interval: int = 50
    eval_interval: int = 2000
    save_interval: int = 5000
    eval_iters: int = 30
    telemetry_interval: int = 2500
    # Compile
    use_compile: bool = True   # torch.compile — big speedup on A100/H100
    # Output
    output_dir: str = "checkpoints"
    run_name: str = "french"
    # Telemetry sentences
    telemetry_sentences: List[str] = field(default_factory=lambda: [
        "<F:en>The European Parliament voted on this resolution<T:fr>Le parlement européen a voté cette résolution",
        "<F:en>The price in euros was fifty pounds<T:fr>Le prix en euros était de cinquante livres",
        "<F:en>Germany and France signed the treaty<T:fr>L'Allemagne et la France ont signé le traité",
    ])


# %% Cell 5 — Training Loop

def get_lr(it, cfg):
    if it < cfg.warmup_iters:
        return cfg.learning_rate * it / max(cfg.warmup_iters, 1)
    if it >= cfg.max_iters:
        return cfg.min_lr
    ratio = (it - cfg.warmup_iters) / (cfg.max_iters - cfg.warmup_iters)
    return cfg.min_lr + 0.5 * (1 + math.cos(math.pi * ratio)) * (cfg.learning_rate - cfg.min_lr)


@torch.no_grad()
def estimate_loss(model, train_ds, val_ds, cfg, ctx):
    was_training = model.training
    model.eval()
    losses = {}
    for split, ds in [("train", train_ds), ("val", val_ds)]:
        total = 0.0
        for _ in range(cfg.eval_iters):
            x, y = ds.get_batch(cfg.batch_size, DEVICE)
            with ctx:
                _, loss = model(x, y)
            total += loss.item()
        losses[split] = total / cfg.eval_iters
    if was_training:
        model.train()
    return losses


@torch.no_grad()
def probe_sparsity(model, sentences, device):
    """Measure sparsity on fixed sentences. Quick telemetry probe."""
    was_training = model.training
    model.eval()
    results = []

    for sent in sentences:
        tokens = torch.tensor([list(sent.encode('utf-8')[:512])], dtype=torch.long, device=device)
        with model.extracting() as buf:
            model(tokens)

        layers = {}
        for L, data in buf.items():
            x_sp = compute_sparsity(data['x'])
            y_sp = compute_sparsity(data['y'])
            layers[str(L)] = {  # str keys for JSON
                'x_sparsity': round(x_sp, 4),
                'y_sparsity': round(y_sp, 4),
            }
        results.append({
            'text_preview': sent[:50],
            'layers': layers,
            'mean_x_sp': round(np.mean([v['x_sparsity'] for v in layers.values()]), 4),
            'mean_y_sp': round(np.mean([v['y_sparsity'] for v in layers.values()]), 4),
        })

    if was_training:
        model.train()
    return results


def save_ckpt(model, optimizer, cfg, it, losses, out_dir, is_best=False):
    ckpt = {
        'iteration': it,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'config': {
            'n_layer': cfg.n_layer, 'n_embd': cfg.n_embd, 'n_head': cfg.n_head,
            'mlp_dim_mult': cfg.mlp_dim_mult, 'dropout': cfg.dropout,
            'vocab_size': cfg.vocab_size,
        },
        'losses': losses,
    }
    torch.save(ckpt, out_dir / f"checkpoint_{it:06d}.pt")
    torch.save(ckpt, out_dir / "latest.pt")
    if is_best:
        torch.save(ckpt, out_dir / "best.pt")


def train(cfg: TrainConfig):
    C = BDHConfig(
        n_layer=cfg.n_layer, n_embd=cfg.n_embd, n_head=cfg.n_head,
        mlp_dim_mult=cfg.mlp_dim_mult, dropout=cfg.dropout, vocab_size=cfg.vocab_size,
    )
    tok_per_iter = cfg.batch_size * cfg.block_size * cfg.gradient_accumulation_steps
    total_tokens = tok_per_iter * cfg.max_iters

    print("=" * 70)
    print(f" BDH Training: {cfg.run_name}")
    print(f"   {C.n_layer}L / {C.n_embd}D / {C.n_head}H / N={C.N}/head ({C.N_total} total)")
    print(f"   batch={cfg.batch_size} × accum={cfg.gradient_accumulation_steps} × T={cfg.block_size} = {tok_per_iter:,} tok/iter")
    print(f"   {cfg.max_iters:,} iters → {total_tokens/1e9:.2f}B token-steps")
    print(f"   torch.compile: {cfg.use_compile}")
    print("=" * 70)

    # Precision
    ptdtype = torch.bfloat16 if DTYPE == "bfloat16" else torch.float16
    ctx = torch.amp.autocast(device_type="cuda", dtype=ptdtype)
    scaler = torch.amp.GradScaler(device="cuda", enabled=(DTYPE == "float16"))

    # Output dirs
    out_dir = Path(cfg.output_dir) / cfg.run_name
    out_dir.mkdir(parents=True, exist_ok=True)
    telem_dir = Path("training_telemetry")
    telem_dir.mkdir(exist_ok=True)

    # Data
    print("\n Data:")
    train_ds = ByteDataset(cfg.train_data, cfg.block_size)
    val_ds = ByteDataset(cfg.val_data, cfg.block_size)

    # Model
    model = BDH(C).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"\n Model: {n_params:,} params ({n_params/1e6:.1f}M)")

    # Compile (big speedup on A100/H100, skip on older GPUs)
    model_for_train = model
    if cfg.use_compile:
        try:
            model_for_train = torch.compile(model)
            print("    torch.compile enabled")
        except Exception as e:
            print(f"    torch.compile failed ({e}), using eager mode")
            model_for_train = model

    # Optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay, betas=(0.9, 0.95),
    )

    # Telemetry accumulator
    evolution = []
    best_val = float('inf')


    # MAIN TRAINING LOOP — with tqdm progress bar

    print(f"\n Training for {cfg.max_iters:,} iterations...\n")
    model.train()

    pbar = tqdm(range(cfg.max_iters), desc=cfg.run_name, unit="it",
                bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}] loss={postfix}')
    pbar.set_postfix_str("...")

    running_loss = 0.0
    running_count = 0
    t_start = time.time()

    for it in pbar:
        # Learning rate schedule
        lr = get_lr(it, cfg)
        for pg in optimizer.param_groups:
            pg['lr'] = lr

        # Forward + backward with gradient accumulation
        optimizer.zero_grad(set_to_none=True)

        accum_loss = 0.0
        for micro in range(cfg.gradient_accumulation_steps):
            x, y = train_ds.get_batch(cfg.batch_size, DEVICE)
            with ctx:
                _, loss = model_for_train(x, y)
                loss = loss / cfg.gradient_accumulation_steps
            scaler.scale(loss).backward()
            accum_loss += loss.item()

        # Gradient clipping + optimizer step
        if cfg.grad_clip > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()

        current_loss = accum_loss * cfg.gradient_accumulation_steps

        # NaN recovery
        if math.isnan(current_loss) or math.isinf(current_loss):
            pbar.write(f" NaN at iter {it}! Restoring best checkpoint...")
            best_ckpt = out_dir / "best.pt"
            if best_ckpt.exists():
                ckpt = torch.load(best_ckpt, map_location=DEVICE, weights_only=False)
                model.load_state_dict(ckpt['model_state_dict'])
                optimizer.load_state_dict(ckpt['optimizer_state_dict'])
                model.train()
            continue

        running_loss += current_loss
        running_count += 1

        # Update progress bar
        if running_count > 0:
            avg = running_loss / running_count
            elapsed = time.time() - t_start
            tps = (running_count * tok_per_iter) / max(elapsed, 1)
            pbar.set_postfix_str(f"{avg:.3f} | lr={lr:.1e} | {tps/1000:.0f}K tok/s")

        # Reset running stats periodically
        if running_count >= cfg.log_interval:
            running_loss = 0.0
            running_count = 0
            t_start = time.time()

        # ── Evaluation + Checkpoint ──
        if it > 0 and it % cfg.eval_interval == 0:
            losses = estimate_loss(model, train_ds, val_ds, cfg, ctx)
            is_best = losses['val'] < best_val
            if is_best:
                best_val = losses['val']
            pbar.write(f"  iter {it}: train={losses['train']:.4f} val={losses['val']:.4f}"
                       f"{'  BEST' if is_best else ''}")
            if it % cfg.save_interval == 0 or is_best:
                save_ckpt(model, optimizer, cfg, it, losses, out_dir, is_best)

        # ── Telemetry probe ──
        if it % cfg.telemetry_interval == 0:
            sp = probe_sparsity(model, cfg.telemetry_sentences, DEVICE)
            entry = {
                'iteration': it,
                'loss': round(current_loss, 4),
                'sparsity': sp,
                'mean_x_sp': round(np.mean([s['mean_x_sp'] for s in sp]), 4),
                'mean_y_sp': round(np.mean([s['mean_y_sp'] for s in sp]), 4),
            }
            evolution.append(entry)
            pbar.write(f"   Sparsity@{it}: x={entry['mean_x_sp']:.1%} y={entry['mean_y_sp']:.1%}")

    pbar.close()

    # Final eval + save
    print("\n Training complete!")
    losses = estimate_loss(model, train_ds, val_ds, cfg, ctx)
    is_best = losses['val'] < best_val
    save_ckpt(model, optimizer, cfg, cfg.max_iters, losses, out_dir, is_best)
    print(f"   Final: train={losses['train']:.4f} val={losses['val']:.4f}")

    # Save evolution
    evo_path = telem_dir / f"evolution_{cfg.run_name}.json"
    with open(evo_path, 'w') as f:
        json.dump(evolution, f, indent=2)
    print(f"    Evolution: {evo_path} ({len(evolution)} points)")

    # Test generation
    print("\n Sample generation:")
    model.eval()
    lang = "fr" if "fr" in cfg.run_name.lower() else "pt"
    prompts = [
        f"<F:en>The European Parliament<T:{lang}>",
        f"<F:en>Economic growth is important<T:{lang}>",
    ]
    for p in prompts:
        tok = torch.tensor([list(p.encode('utf-8'))], dtype=torch.long, device=DEVICE)
        with torch.no_grad():
            out = model.generate(tok, max_new_tokens=80, top_k=5, temperature=0.8)
        text = bytes(out[0].cpu().tolist()).decode('utf-8', errors='replace')
        print(f"  {text[:120]}")

    # Sparsity check
    sp = probe_sparsity(model, cfg.telemetry_sentences[:1], DEVICE)
    print(f"\n Final sparsity: x={sp[0]['mean_x_sp']:.1%}, y={sp[0]['mean_y_sp']:.1%}")

    return model


# %% Cell 6 — TRAIN FRENCH (~45-60 min on A100)
french_model = train(TrainConfig(
    train_data="data/en-fr/train.bin",
    val_data="data/en-fr/val.bin",
    run_name="french",
    max_iters=50000,
))


# %% Cell 7 — TRAIN PORTUGUESE (~35-45 min on A100)
pt_model = train(TrainConfig(
    train_data="data/en-pt/train.bin",
    val_data="data/en-pt/val.bin",
    run_name="portuguese",
    max_iters=40000,
    telemetry_sentences=[
        "<F:en>The European Parliament voted on this resolution<T:pt>O parlamento europeu votou esta resolução",
        "<F:en>The price in euros was fifty pounds<T:pt>O preço em euros foi de cinquenta libras",
        "<F:en>Germany and France signed the treaty<T:pt>A Alemanha e a França assinaram o tratado",
    ],
))


# %% Cell 8 — MERGE MODELS (Paper §7.1)
print("\n" + "=" * 70)
print(" Merging French + Portuguese (Paper §7.1: neuron concatenation)")
print("=" * 70)

def merge_models(path_a, path_b, output_path):
    """Merge by concatenating along neuron dimension. Shared weights are averaged."""
    ckpt_a = torch.load(path_a, map_location='cpu', weights_only=False)
    ckpt_b = torch.load(path_b, map_location='cpu', weights_only=False)

    def clean(sd):
        return {k.replace('_orig_mod.', ''): v for k, v in sd.items()}

    sd_a, sd_b = clean(ckpt_a['model_state_dict']), clean(ckpt_b['model_state_dict'])
    cfg = ckpt_a['config']

    merged = {}

    # Average shared weights
    for key in ['embed.weight', 'lm_head', 'pos_emb.weight']:
        if key in sd_a and key in sd_b:
            merged[key] = (sd_a[key] + sd_b[key]) / 2.0

    # Copy buffers
    for key in sd_a:
        if 'rope_freqs' in key or 'ln.' in key:
            merged[key] = sd_a[key]

    # Concatenate along neuron dimension
    merged['encoder'] = torch.cat([sd_a['encoder'], sd_b['encoder']], dim=0)  # (N_a+N_b, D)
    merged['decoder_x'] = torch.cat([sd_a['decoder_x'], sd_b['decoder_x']], dim=2)  # (H,D,N_a+N_b)
    merged['decoder_y'] = torch.cat([sd_a['decoder_y'], sd_b['decoder_y']], dim=2)

    # Update config
    orig_N_per_head = cfg['n_embd'] * cfg['mlp_dim_mult'] // cfg['n_head']
    merged_cfg = dict(cfg)
    merged_cfg['mlp_dim_mult'] = cfg['mlp_dim_mult'] * 2  # Doubled neurons

    heritage = {
        'model_a': str(path_a), 'model_b': str(path_b),
        'neurons_a': orig_N_per_head * cfg['n_head'],
        'neurons_b': orig_N_per_head * cfg['n_head'],
        'total': orig_N_per_head * 2 * cfg['n_head'],
    }

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        'config': merged_cfg, 'model_state_dict': merged,
        'heritage': heritage, 'losses': {},
    }, output_path)

    print(f"   Saved: {output_path}")
    print(f"     Neurons: {heritage['neurons_a']} (FR) + {heritage['neurons_b']} (PT) = {heritage['total']}")
    return heritage


fr_path = Path("checkpoints/french/best.pt")
pt_path = Path("checkpoints/portuguese/best.pt")
merge_path = Path("checkpoints/merged/merged.pt")

if fr_path.exists() and pt_path.exists():
    heritage = merge_models(fr_path, pt_path, merge_path)
else:
    missing = [p for p in [fr_path, pt_path] if not p.exists()]
    print(f"   Missing: {missing}")


# %% Cell 9 — Evaluate Merged Model
print("\n Evaluating all models...")

def eval_model(model_path, data_path, n_batches=30):
    ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
    cfg = BDHConfig(**ckpt['config'])
    m = BDH(cfg).to(DEVICE)
    sd = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state_dict'].items()}
    m.load_state_dict(sd, strict=False)
    m.eval()
    ds = ByteDataset(data_path, 256)
    ctx = torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16)
    total = 0.0
    for _ in range(n_batches):
        x, y = ds.get_batch(8, DEVICE)
        with ctx, torch.no_grad():
            _, loss = m(x, y)
        total += loss.item()
    del m
    torch.cuda.empty_cache()
    return total / n_batches


eval_results = {}
models = {'french': fr_path, 'portuguese': pt_path, 'merged': merge_path}
datasets = {'fr_data': 'data/en-fr/val.bin', 'pt_data': 'data/en-pt/val.bin'}

for mname, mpath in models.items():
    if not mpath.exists():
        continue
    eval_results[mname] = {}
    for dname, dpath in datasets.items():
        if not Path(dpath).exists():
            continue
        loss = eval_model(mpath, dpath)
        eval_results[mname][dname] = round(loss, 4)
        print(f"  {mname:12s} on {dname}: {loss:.4f}")

# Generate samples
samples = []
if merge_path.exists():
    ckpt = torch.load(merge_path, map_location=DEVICE, weights_only=False)
    cfg = BDHConfig(**ckpt['config'])
    merged_m = BDH(cfg).to(DEVICE)
    sd = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state_dict'].items()}
    merged_m.load_state_dict(sd, strict=False)
    merged_m.eval()

    prompts = [
        "<F:en>The European Parliament<T:fr>",
        "<F:en>The European Parliament<T:pt>",
        "<F:en>Economic growth is essential<T:fr>",
        "<F:en>Economic growth is essential<T:pt>",
        "<F:fr>Le parlement européen<T:en>",
        "<F:pt>O parlamento europeu<T:en>",
    ]
    print("\n Merged model samples:")
    for p in prompts:
        tok = torch.tensor([list(p.encode('utf-8'))], dtype=torch.long, device=DEVICE)
        with torch.no_grad():
            out = merged_m.generate(tok, max_new_tokens=80, top_k=5)
        text = bytes(out[0].cpu().tolist()).decode('utf-8', errors='replace')
        samples.append({'prompt': p, 'output': text})
        print(f"  {text[:100]}")
    del merged_m
    torch.cuda.empty_cache()

# Save merge data
telem_dir = Path("training_telemetry")
telem_dir.mkdir(exist_ok=True)
merge_data = {
    'evaluation': eval_results,
    'heritage': heritage if 'heritage' in dir() else {},
    'samples': samples,
}
with open(telem_dir / "merge_eval.json", 'w') as f:
    json.dump(merge_data, f, indent=2)
print(" Merge evaluation saved")


# %% Cell 10 — Package for Download
print("\n Packaging...")
pkg = Path("kriti_outputs")
pkg.mkdir(exist_ok=True)

for name in ["french", "portuguese", "merged"]:
    for fname in ["best.pt", "latest.pt", "merged.pt"]:
        src = Path(f"checkpoints/{name}/{fname}")
        if src.exists():
            dst = pkg / f"{name}_{fname}"
            shutil.copy2(src, dst)
            print(f"   {dst.name} ({dst.stat().st_size/1e6:.1f}MB)")
            break

for f in Path("training_telemetry").glob("*.json"):
    shutil.copy2(f, pkg / f.name)
    print(f"   {f.name}")

shutil.make_archive("kriti_checkpoints", 'zip', str(pkg))
print(f"\n kriti_checkpoints.zip ready!")

try:
    from google.colab import files
    files.download("kriti_checkpoints.zip")
except ImportError:
    print("(Download manually)")

print("""
Done
""")

PyTorch 2.10.0+cu128
GPU: NVIDIA H100 80GB HBM3 (85GB)
Precision: bfloat16
 Model sanity check passed. Output shape: torch.Size([2, 64, 256]), Loss: 78044.211
   Params: 7,962,624 (8.0M)
   Config: 6L/192D/4H, N=3072/head, N_total=12288
 French data:


  DL: 24746it [00:28, 868.40it/s]                            
/tmp/ipython-input-69421997.py:283: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(data_dir)


  Extracting...
  Pairs: 2,007,723


  Building: 100%|██████████| 2007723/2007723 [00:01<00:00, 1093498.55it/s]


  train.bin: 655.8MB
  val.bin: 34.5MB

 Portuguese data:


...

portuguese:  95%|█████████▍| 37999/40000 [57:32<02:59, 11.14it/s] loss=, 1.393 | lr=1.1e-04 | 33K tok/s

  iter 38000: train=0.6777 val=0.6932  BEST


portuguese: 100%|██████████| 40000/40000 [1:00:30<00:00, 11.02it/s] loss=, 1.380 | lr=1.0e-04 | 368K tok/s



 Training complete!
   Final: train=0.6798 val=0.6961
    Evolution: training_telemetry/evolution_portuguese.json (16 points)

 Sample generation:
  <F:en>The European Parliament<T:pt>ExecuÃ§Ã£o do Parlamento<F:en>Mr President, I would like to congratulate the
  <F:en>Economic growth is important<T:pt>Assunto: EconÃ³mico importante Ã© a de abertura.<F:en>We are still faced wit

 Final sparsity: x=95.0%, y=99.0%

 Merging French + Portuguese (Paper §7.1: neuron concatenation)
   Saved: checkpoints/merged/merged.pt
     Neurons: 12288 (FR) + 12288 (PT) = 24576

 Evaluating all models...
  data/en-fr/val.bin: 34.5MB
  french       on fr_data: 0.6971
  data/en-pt/val.bin: 33.2MB
  french       on pt_data: 2.2853
  data/en-fr/val.bin: 34.5MB
  portuguese   on fr_data: 2.3660
  data/en-pt/val.bin: 33.2MB
  portuguese   on pt_data: 0.7317


RuntimeError: Error(s) in loading state_dict for BDH:
	size mismatch for rope_freqs: copying a param with shape torch.Size([1, 1, 1, 3072]) from checkpoint, the shape in current model is torch.Size([1, 1, 1, 6144]).

In [ ]:
import torch
import shutil
import json
from pathlib import Path

print("\n" + "=" * 70)
print("  FIXING AND MERGING MODELS (Corrected RoPE)")
print("=" * 70)

def fix_and_merge_models(path_a, path_b, output_path):
    ckpt_a = torch.load(path_a, map_location='cpu', weights_only=False)
    ckpt_b = torch.load(path_b, map_location='cpu', weights_only=False)

    def clean(sd):
        return {k.replace('_orig_mod.', ''): v for k, v in sd.items()}

    sd_a, sd_b = clean(ckpt_a['model_state_dict']), clean(ckpt_b['model_state_dict'])
    cfg = ckpt_a['config']
    merged = {}

    # Average shared weights
    for key in ['embed.weight', 'lm_head', 'pos_emb.weight']:
        if key in sd_a and key in sd_b:
            merged[key] = (sd_a[key] + sd_b[key]) / 2.0

    # Copy LayerNorm
    for key in sd_a:
        if 'ln.' in key:
            merged[key] = sd_a[key]

    # Concatenate along neuron dimension
    merged['encoder'] = torch.cat([sd_a['encoder'], sd_b['encoder']], dim=0)
    merged['decoder_x'] = torch.cat([sd_a['decoder_x'], sd_b['decoder_x']], dim=2)
    merged['decoder_y'] = torch.cat([sd_a['decoder_y'], sd_b['decoder_y']], dim=2)


    if 'rope_freqs' in merged:
        del merged['rope_freqs']
    # ---------------------------------------------------------

    # Update config
    orig_N_per_head = cfg['n_embd'] * cfg['mlp_dim_mult'] // cfg['n_head']
    merged_cfg = dict(cfg)
    merged_cfg['mlp_dim_mult'] = cfg['mlp_dim_mult'] * 2

    heritage = {
        'model_a': str(path_a), 'model_b': str(path_b),
        'neurons_a': orig_N_per_head * cfg['n_head'],
        'neurons_b': orig_N_per_head * cfg['n_head'],
        'total': orig_N_per_head * 2 * cfg['n_head'],
    }

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        'config': merged_cfg, 'model_state_dict': merged,
        'heritage': heritage, 'losses': {},
    }, output_path)

    print(f"   Saved Fixed Merged Model: {output_path}")
    return heritage

fr_path = Path("checkpoints/french/best.pt")
pt_path = Path("checkpoints/portuguese/best.pt")
merge_path = Path("checkpoints/merged/merged.pt")

if fr_path.exists() and pt_path.exists():
    heritage = fix_and_merge_models(fr_path, pt_path, merge_path)
else:
    print("Error: Missing base checkpoints!")

# --- EVALUATION ---
print("\n Evaluating Fixed Merged Model...")

def eval_model(model_path, data_path, n_batches=30):
    ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
    cfg = BDHConfig(**ckpt['config'])
    m = BDH(cfg).to(DEVICE)
    sd = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state_dict'].items()}
    # strict=False allows it to ignore the missing rope_freqs and use its own
    m.load_state_dict(sd, strict=False)
    m.eval()
    ds = ByteDataset(data_path, 256)
    ctx = torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16)
    total = 0.0
    for _ in range(n_batches):
        x, y = ds.get_batch(8, DEVICE)
        with ctx, torch.no_grad():
            _, loss = m(x, y)
        total += loss.item()
    del m
    torch.cuda.empty_cache()
    return total / n_batches

if merge_path.exists():
    eval_results = {'merged': {}}
    datasets = {'fr_data': 'data/en-fr/val.bin', 'pt_data': 'data/en-pt/val.bin'}

    for dname, dpath in datasets.items():
        if Path(dpath).exists():
            loss = eval_model(merge_path, dpath)
            eval_results['merged'][dname] = round(loss, 4)
            print(f"  merged on {dname}: {loss:.4f}")

    # Generate samples
    ckpt = torch.load(merge_path, map_location=DEVICE, weights_only=False)
    cfg = BDHConfig(**ckpt['config'])
    merged_m = BDH(cfg).to(DEVICE)
    sd = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state_dict'].items()}
    merged_m.load_state_dict(sd, strict=False)
    merged_m.eval()

    samples = []
    prompts = [
        "<F:en>The European Parliament<T:fr>",
        "<F:en>The European Parliament<T:pt>",
    ]
    print("\n Merged model samples:")
    for p in prompts:
        tok = torch.tensor([list(p.encode('utf-8'))], dtype=torch.long, device=DEVICE)
        with torch.no_grad():
            out = merged_m.generate(tok, max_new_tokens=80, top_k=5)
        text = bytes(out[0].cpu().tolist()).decode('utf-8', errors='replace')
        samples.append({'prompt': p, 'output': text})
        print(f"  {text[:100]}")

    # Save data
    telem_dir = Path("training_telemetry")
    telem_dir.mkdir(exist_ok=True)

    # We load the existing eval results for French/Portuguese and just append the Merged ones
    merge_data = {'evaluation': {}}
    if (telem_dir / "merge_eval.json").exists():
        with open(telem_dir / "merge_eval.json", 'r') as f:
            merge_data = json.load(f)

    merge_data['evaluation']['merged'] = eval_results['merged']
    merge_data['heritage'] = heritage
    merge_data['samples'] = samples

    with open(telem_dir / "merge_eval.json", 'w') as f:
        json.dump(merge_data, f, indent=2)
    print(" Merge evaluation saved")

# --- PACKAGE ---
print("\n Packaging...")
pkg = Path("kriti_outputs")
pkg.mkdir(exist_ok=True)

for name in ["french", "portuguese", "merged"]:
    for fname in ["best.pt", "latest.pt", "merged.pt"]:
        src = Path(f"checkpoints/{name}/{fname}")
        if src.exists():
            dst = pkg / f"{name}_{fname}"
            shutil.copy2(src, dst)

for f in Path("training_telemetry").glob("*.json"):
    shutil.copy2(f, pkg / f.name)

shutil.make_archive("kriti_checkpoints", 'zip', str(pkg))
print(f"\n kriti_checkpoints.zip ready!")

try:
    from google.colab import files
    files.download("kriti_checkpoints.zip")
except:
    pass


  FIXING AND MERGING MODELS (Corrected RoPE)
   Saved Fixed Merged Model: checkpoints/merged/merged.pt

 Evaluating Fixed Merged Model...
  data/en-fr/val.bin: 34.5MB
  merged on fr_data: 3276.8300
  data/en-pt/val.bin: 33.2MB
  merged on pt_data: 3613.6104

 Merged model samples:
  <F:en>The European Parliament<T:fr>iiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii
  <F:en>The European Parliament<T:pt>iiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii
 Merge evaluation saved

 Packaging...

 kriti_checkpoints.zip ready!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

!pip uninstall -y community python-louvain

# 2. Reinstall ONLY the correct graph clustering package
!pip install python-louvain

# 3. Force Python to reload the dictionary so it finds the right one
import community
import importlib
importlib.reload(community)
import community as community_louvain

print(" Namespace fixed! You can now re-run the extraction cell.")

Found existing installation: community 1.0.0b1
Uninstalling community-1.0.0b1:
  Successfully uninstalled community-1.0.0b1
Found existing installation: python-louvain 0.16
Uninstalling python-louvain-0.16:
  Successfully uninstalled python-louvain-0.16
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 19.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for python-louvain: filename=python_louvain-0.16-py3-none-any.whl size=9388 sha256=9bb2f1c936e5f7a5051bff1fac3a2ded125e3ad6ec42196ef4dd31bb13718871
  Stored in directory: /root/.cache/pip/wheels/40/f1/e3/485b698c520fa0baee1d07897abc7b8d6479b7d199ce96f4af
Successfully built python-louvain


 Namespace fixed! You can now re-run the extraction cell.


In [ ]:


# %% Cell 1 — Setup
import subprocess, sys, os
def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
install("torch"); install("numpy"); install("tqdm")
install("networkx"); install("python-louvain")

import torch, torch.nn.functional as F, numpy as np, math, json, shutil
from torch import nn, Tensor
from dataclasses import dataclass
from typing import Optional, Dict, List
from contextlib import contextmanager
from pathlib import Path
from tqdm import tqdm
import networkx as nx
import community as community_louvain

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


# %% Cell 2 — Model Architecture (MUST match Train v3 exactly!)

@dataclass
class BDHConfig:
    n_layer: int = 6
    n_embd: int = 192
    n_head: int = 4
    dropout: float = 0.1
    vocab_size: int = 256
    mlp_dim_mult: int = 64
    @property
    def N(self): return self.n_embd * self.mlp_dim_mult // self.n_head
    @property
    def N_total(self): return self.N * self.n_head


class BDH(nn.Module):
    """Must match Train v3 architecture exactly."""
    def __init__(self, config):
        super().__init__()
        self.config = config
        D, H, N = config.n_embd, config.n_head, config.N
        self.embed = nn.Embedding(config.vocab_size, D)
        self.pos_emb = nn.Embedding(4096, D)
        self.ln = nn.LayerNorm(D, elementwise_affine=False, bias=False)
        self.drop = nn.Dropout(config.dropout)
        self.encoder = nn.Parameter(torch.empty(H*N, D))
        self.decoder_x = nn.Parameter(torch.empty(H, D, N))
        self.decoder_y = nn.Parameter(torch.empty(H, D, N))
        self.lm_head = nn.Parameter(torch.empty(D, config.vocab_size))
        freqs = self._build_rope_freqs(N)
        self.register_buffer('rope_freqs', freqs)
        self._extract = False
        self._buffer = {}

    def _build_rope_freqs(self, N, theta=2**16):
        idx = torch.arange(N, dtype=torch.float32)
        idx_q = (idx / 2).floor() * 2
        return (1.0 / (theta ** (idx_q / N)) / (2 * math.pi)).view(1,1,1,N)

    def _causal_attention(self, Q, K, V):
        B, H, T, N = Q.size()
        positions = torch.arange(T, device=Q.device, dtype=Q.dtype).view(1,1,T,1)
        phases = (positions * self.rope_freqs[:,:,:,:N].to(Q.dtype) % 1.0) * (2*math.pi)
        cos_p, sin_p = torch.cos(phases), torch.sin(phases)
        Q_rot = torch.stack((-Q[...,1::2], Q[...,::2]), dim=-1).reshape_as(Q)
        Q_roped = Q * cos_p + Q_rot * sin_p
        scores = torch.matmul(Q_roped, Q_roped.transpose(-2,-1))
        mask = torch.tril(torch.ones(T,T,device=Q.device,dtype=torch.bool), diagonal=-1)
        scores = scores.masked_fill(~mask, 0.0)
        return torch.matmul(scores, V), scores

    @contextmanager
    def extracting(self):
        self._extract = True; self._buffer = {}
        try: yield self._buffer
        finally: self._extract = False; self._buffer = {}

    def forward(self, idx, targets=None):
        C = self.config; B, T = idx.size()
        D, H, N = C.n_embd, C.n_head, C.N
        v_ast = (self.embed(idx) + self.pos_emb(torch.arange(T, device=idx.device))).unsqueeze(1)
        for L in range(C.n_layer):
            v_n = self.ln(v_ast.squeeze(1)).unsqueeze(1)
            x_pre = torch.einsum('bitd,hdn->bhtn', v_n, self.decoder_x)
            x = F.relu(x_pre)
            a_ast, attn = self._causal_attention(x, x, v_n.expand(B,H,T,D))
            y_pre = torch.einsum('bhtd,hdn->bhtn', a_ast, self.decoder_y)
            y = F.relu(y_pre) * x
            if self._extract:
                self._buffer[L] = {
                    'x_pre': x_pre.detach().cpu(), 'x': x.detach().cpu(),
                    'y_pre': y_pre.detach().cpu(), 'y': y.detach().cpu(),
                    'a_ast': a_ast.detach().cpu(), 'v_ast': v_n.detach().cpu(),
                    'attn': attn.detach().cpu(),
                }
            y_f = self.drop(y.permute(0,2,1,3).reshape(B,T,H*N))
            v_ast = v_ast + torch.matmul(y_f, self.encoder).unsqueeze(1)
        logits = torch.matmul(v_ast.squeeze(1), self.lm_head)
        loss = F.cross_entropy(logits.view(-1,C.vocab_size), targets.view(-1)) if targets is not None else None
        return logits, loss


def load_model(path, device='cpu'):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    cfg = BDHConfig(**ckpt['config'])
    model = BDH(cfg)
    sd = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state_dict'].items()}
    model.load_state_dict(sd, strict=False)
    return model.to(device).eval(), cfg, ckpt

def sparsity(x):
    return 1.0 - ((x > 0).sum().item() / max(x.numel(), 1))


# %% Cell 3 — Analysis Corpus

CORPUS = [
    # Currency
    "<F:en>The price was fifty euros and thirty pounds<T:fr>Le prix était de cinquante euros et trente livres",
    "<F:en>The dollar strengthened against the yen today<T:fr>Le dollar s'est renforcé face au yen aujourd'hui",
    "<F:en>Converting francs to marks was expensive<T:fr>Convertir les francs en marks était coûteux",
    # Country
    "<F:en>Germany and France signed the bilateral treaty<T:fr>L'Allemagne et la France ont signé le traité bilatéral",
    "<F:en>Sweden and Finland proposed a new allocation<T:fr>La Suède et la Finlande ont proposé une nouvelle répartition",
    "<F:en>The delegation from Portugal arrived in Brussels<T:fr>La délégation du Portugal est arrivée à Bruxelles",
    # Political
    "<F:en>The European Parliament voted on this resolution<T:fr>Le parlement européen a voté cette résolution",
    "<F:en>The Commission presented the annual budget report<T:fr>La Commission a présenté le rapport budgétaire annuel",
    "<F:en>The Council of Ministers reached a compromise<T:fr>Le Conseil des ministres est parvenu à un compromis",
    # Syntactic variety
    "<F:en>What measures has the Commission taken to address this issue?<T:fr>Quelles mesures la Commission a-t-elle prises?",
    "<F:en>The amendment proposed by Mr Ferber was adopted<T:fr>L'amendement proposé par M. Ferber a été adopté",
    # Repetition (Hebbian strengthening)
    "<F:en>The budget was discussed. The budget was approved. The budget will be implemented.<T:fr>Le budget a été discuté. Le budget a été approuvé. Le budget sera mis en œuvre.",
    # Short
    "<F:en>Good morning<T:fr>Bonjour",
    "<F:en>Thank you very much<T:fr>Merci beaucoup",
    # Mixed currency+country
    "<F:en>The British pound fell after the vote in London<T:fr>La livre sterling a chuté après le vote à Londres",
]

CONCEPT_BANK = {
    "currency": [
        "The price was fifty euros", "The dollar weakened today",
        "Converting pounds to francs", "The yen exchange rate",
        "Payment in marks was accepted", "The cost in euros was high",
        "Sterling fell sharply", "The franc to dollar rate",
    ],
    "country": [
        "Germany signed the agreement", "France proposed the amendment",
        "Sweden joined the coalition", "Finland supported the motion",
        "Portugal sent a delegation", "The Spanish representative spoke",
        "Italy abstained from voting", "The Greek economy recovered",
    ],
    "institution": [
        "The European Parliament voted", "The Commission published a report",
        "The Council reached agreement", "The Committee examined the proposal",
        "Parliament debated the resolution", "The Commission responded",
    ],
    "action_verb": [
        "The assembly voted unanimously", "The ministers signed the treaty",
        "The delegation proposed amendments", "The committee discussed the budget",
        "The rapporteur presented findings", "The parliament adopted the resolution",
    ],
}


# %% Cell 4 — Load Model

CKPT = "checkpoints/french/best.pt"
print(f" Loading {CKPT}...")
model, config, ckpt = load_model(CKPT, DEVICE)
print(f"  {config.n_layer}L/{config.n_embd}D/{config.n_head}H, N={config.N}/head")
print(f"  Params: {sum(p.numel() for p in model.parameters()):,}")

OUT = Path("viz_data")
for d in ["telemetry","graph","synapses","monosemanticity","evolution","merge","sparsity","hero_tokens"]:
    (OUT/d).mkdir(parents=True, exist_ok=True)

# Meta
json.dump({
    'config': {'n_layer':config.n_layer, 'n_embd':config.n_embd, 'n_head':config.n_head,
               'N':config.N, 'N_total':config.N_total},
    'n_params': sum(p.numel() for p in model.parameters()),
    'iteration': ckpt.get('iteration','?'),
    'losses': ckpt.get('losses',{}),
    'equations': {
        'x_sparse': 'x_{t,l} = ReLU(D_x · LN(v*_{t,l-1}))',
        'attention': 'a*_{t,l} = CausalAttn(Q=x, K=x, V=v*)',
        'y_gated': 'y_{t,l} = ReLU(D_y · a*_{t,l}) ⊙ x_{t,l}',
        'update': 'v*_{t,l} = v*_{t,l-1} + E · y_{t,l}',
    },
}, open(OUT/"meta.json",'w'), indent=2)


# %% Cell 5 — Per-Token Telemetry (THE CORE)

print("\n" + "="*65)
print(" PHASE 1: Per-token telemetry")
print("="*65)

HERO_WORDS = {"euro","dollar","pound","yen","franc","mark","Germany","France",
              "Sweden","budget","Parliament","Commission","Portugal","London"}

def extract_sentence(model, text, idx, device):
    raw = list(text.encode('utf-8'))
    tokens = torch.tensor([raw], dtype=torch.long, device=device)
    T = len(raw)
    chars = [chr(b) if 32<=b<127 else f"\\x{b:02x}" for b in raw]

    with torch.no_grad(), model.extracting() as buf:
        logits, _ = model(tokens)

    token_data = []
    heroes = []

    for t in range(T):
        # Check if this char is part of a hero word
        context = ''.join(chars[max(0,t-6):t+7])
        is_hero = any(w in context for w in HERO_WORDS)

        entry = {'idx':t, 'byte':raw[t], 'char':chars[t], 'layers':{}}

        for L in range(config.n_layer):
            d = buf[L]
            x_pre_t = d['x_pre'][0,:,t,:]  # (H,N)
            x_t = d['x'][0,:,t,:]
            y_pre_t = d['y_pre'][0,:,t,:]
            y_t = d['y'][0,:,t,:]

            # Pre-ReLU histogram (the "95% wipeout")
            xp = x_pre_t.flatten().numpy()
            hc, he = np.histogram(xp, bins=50)

            # Active neurons
            x_nz = int((x_t > 0).sum())
            y_nz = int((y_t > 0).sum())
            x_tot = x_t.numel()
            y_tot = y_t.numel()

            # Top active
            xf = x_t.flatten()
            tv, ti = torch.topk(xf, min(30, max(x_nz,1)))
            yf = y_t.flatten()
            yv, yi = torch.topk(yf, min(30, max(y_nz,1)))

            layer = {
                'x_pre_hist': {'counts':hc.tolist(), 'edges':he.tolist(),
                               'frac_neg':round(float((xp<0).mean()),4),
                               'mean':round(float(xp.mean()),6), 'std':round(float(xp.std()),6)},
                'x_sparsity': round(1-x_nz/x_tot,4), 'x_active': x_nz,
                'y_sparsity': round(1-y_nz/y_tot,4), 'y_active': y_nz,
                'x_top': {'idx':ti.tolist(),'val':[round(v,4) for v in tv.tolist()]},
                'y_top': {'idx':yi.tolist(),'val':[round(v,4) for v in yv.tolist()]},
            }

            # y pre-ReLU hist too
            yp = y_pre_t.flatten().numpy()
            yhc, yhe = np.histogram(yp, bins=50)
            layer['y_pre_hist'] = {'counts':yhc.tolist(), 'edges':yhe.tolist(),
                                   'frac_neg':round(float((yp<0).mean()),4)}

            entry['layers'][str(L)] = layer

            if is_hero:
                heroes.append({
                    'sent':idx, 'tok':t, 'char':chars[t], 'layer':L,
                    'x_pre':x_pre_t.numpy().tolist(),
                    'x':x_t.numpy().tolist(),
                    'y':y_t.numpy().tolist(),
                })

        token_data.append(entry)

    # Next-token predictions
    probs = F.softmax(logits[0], dim=-1)
    top_p, top_i = torch.topk(probs, 5, dim=-1)
    preds = []
    for t in range(T):
        p = [{'byte':int(top_i[t,k]),'prob':round(float(top_p[t,k]),4),
              'char':chr(top_i[t,k].item()) if 32<=top_i[t,k]<127 else '?'} for k in range(5)]
        preds.append(p)

    return {'idx':idx, 'text':text, 'n_tokens':T, 'tokens':token_data, 'predictions':preds}, heroes


corpus_meta = []
all_heroes = []

for i, text in enumerate(tqdm(CORPUS, desc="Extracting")):
    result, heroes = extract_sentence(model, text, i, DEVICE)
    json.dump(result, open(OUT/f"telemetry/sentence_{i:02d}.json",'w'))

    mean_sp = np.mean([result['tokens'][t]['layers'][str(L)]['x_sparsity']
                       for t in range(result['n_tokens']) for L in range(config.n_layer)])
    corpus_meta.append({'idx':i, 'text':text, 'n_tokens':result['n_tokens'],
                       'mean_sparsity':round(mean_sp,4)})
    all_heroes.extend(heroes)

json.dump(corpus_meta, open(OUT/"corpus.json",'w'), indent=2)

for i, h in enumerate(all_heroes[:50]):
    json.dump(h, open(OUT/f"hero_tokens/hero_{i:03d}.json",'w'))

print(f" {len(CORPUS)} sentences, {len(all_heroes)} hero dumps")


# %% Cell 6 — Graph Topology (G* = Dₓ · E)

print("\n" + "="*65)
print("  PHASE 2: Graph topology")
print("="*65)

with torch.no_grad():
    D, H, N = config.n_embd, config.n_head, config.N
    E = model.encoder.view(H, N, D)  # (H, N, D)

    for h in range(H):
        # G*[h] = Dₓ[h]ᵀ @ E[h]ᵀ = (N,D) @ (D,N) = (N,N)
        Gstar = torch.mm(model.decoder_x[h].T, E[h].T).cpu().numpy()

        absG = np.abs(Gstar)
        thresh = float(np.percentile(absG, 99))
        binary = (absG >= thresh).astype(float)
        n_edges = int(binary.sum())

        out_deg = binary.sum(axis=1)
        in_deg = binary.sum(axis=0)

        # Louvain
        G_nx = nx.Graph()
        rs, cs = np.where(binary > 0)
        for r, c in zip(rs.tolist(), cs.tolist()):
            if r != c:
                G_nx.add_edge(r, c, weight=float(absG[r,c]))

        if G_nx.number_of_nodes() > 10:
            partition = community_louvain.best_partition(G_nx)
            modularity = community_louvain.modularity(partition, G_nx)
        else:
            partition, modularity = {}, 0.0

        clusters = {}
        for node, cid in partition.items():
            k = str(cid)
            clusters.setdefault(k, []).append(node)

        # Top edges
        edges = sorted([(int(r),int(c),round(float(absG[r,c]),4)) for r,c in zip(rs,cs)],
                       key=lambda e:-e[2])[:500]

        json.dump({
            'head':h, 'N':N, 'n_edges':n_edges, 'threshold':round(thresh,4),
            'modularity':round(modularity,4), 'n_clusters':len(clusters),
            'edges':edges, 'clusters':clusters,
            'out_deg_hist':np.histogram(out_deg, bins=30)[0].tolist(),
            'in_deg_hist':np.histogram(in_deg, bins=30)[0].tolist(),
            'hubs':sorted([(int(i),int(d)) for i,d in enumerate(out_deg) if d>0], key=lambda x:-x[1])[:50],
        }, open(OUT/f"graph/gstar_head{h}.json",'w'))

        print(f"  Head {h}: {n_edges} edges, {len(clusters)} clusters, modularity={modularity:.3f}")

print(" Graph done")


# %% Cell 7 — Hebbian Synapse Tracking

print("\n" + "="*65)
print(" PHASE 3: Synapse tracking")
print("="*65)

# For tracking synapses, we need the RECURRENT form of attention
# to observe ρ accumulation. We do this manually, outside the model.

with torch.no_grad():
    D, H, N = config.n_embd, config.n_head, config.N
    E = model.encoder.view(H, N, D)

    # Identify top synapses from G*
    Gstar_h0 = torch.mm(model.decoder_x[0].T, E[0].T).cpu()
    flat = Gstar_h0.abs().flatten()
    top_v, top_i = torch.topk(flat, 20)
    tracked = [{'id':k, 'i':top_i[k].item()//N, 'j':top_i[k].item()%N,
                'weight':round(float(Gstar_h0.flatten()[top_i[k]]),4)}
               for k in range(20)]

    print(f"  Tracking {len(tracked)} synapses from G*[head=0]")


def track_rho(model, text, tracked_synapses, device):
    """Manually compute ρ accumulation token-by-token to track synapses."""
    raw = list(text.encode('utf-8'))
    tokens = torch.tensor([raw], dtype=torch.long, device=device)
    T = len(raw)

    # Run forward to get x activations at layer 0
    with model.extracting() as buf:
        model(tokens)

    # x at layer 0: (1, H, T, N) — these are the K vectors for ρ
    x_L0 = buf[0]['x'][0, 0, :, :]  # (T, N) for head 0
    v_L0 = buf[0]['v_ast'][0, 0, :, :]  # (T, D)

    # Manual ρ accumulation: ρ[t] = Σ_{τ<t} K[τ]ᵀ V[τ]
    # Tracked synapse σ(i,j) ≈ ρ[i,:] · E[j,:]
    E_h0 = E[0].to(device)  # (N, D)

    rho = torch.zeros(N, D, device=device, dtype=torch.float32)
    timeline = []
    chars = [chr(b) if 32<=b<127 else f"\\x{b:02x}" for b in raw]

    for t in range(T):
        # Record synapse values BEFORE update (what model sees at time t)
        vals = []
        for s in tracked_synapses:
            # σ(i,j) = ρ[i,:] · E[j,:]
            val = float(torch.dot(rho[s['i']], E_h0[s['j']]))
            vals.append(round(val, 6))
        timeline.append({'t':t, 'char':chars[t], 'byte':raw[t], 'vals':vals})

        # Update ρ: ρ += K[t]ᵀ V[t]
        k_t = x_L0[t].to(device).float()  # (N,)
        v_t = v_L0[t].to(device).float()  # (D,)
        rho += k_t.unsqueeze(1) * v_t.unsqueeze(0)  # (N, D)

    return timeline


synapse_data = {'tracked': tracked, 'sentences': []}
track_indices = [0, 1, 3, 6, 11, 14]  # Currency, country, political, repetition, mixed

for idx in track_indices:
    if idx >= len(CORPUS): continue
    text = CORPUS[idx]
    print(f"  Sentence {idx}: {text[:50]}...")
    tl = track_rho(model, text, tracked, DEVICE)
    synapse_data['sentences'].append({'idx':idx, 'text':text, 'timeline':tl})

json.dump(synapse_data, open(OUT/"synapses/timeline.json",'w'))
print(" Synapse tracking done")


# %% Cell 8 — Monosemanticity

print("\n" + "="*65)
print(" PHASE 4: Monosemanticity")
print("="*65)

def neuron_activations(model, text, device):
    tokens = torch.tensor([list(text.encode('utf-8')[:512])], dtype=torch.long, device=device)
    with torch.no_grad(), model.extracting() as buf:
        model(tokens)
    acts = {}
    for L in range(config.n_layer):
        acts[L] = {
            'x': buf[L]['x'].squeeze(0).mean(dim=1).cpu().numpy(),  # (H,N) mean over T
            'y': buf[L]['y'].squeeze(0).mean(dim=1).cpu().numpy(),
        }
    return acts


concept_acts = {}
for concept, sents in CONCEPT_BANK.items():
    xs = []
    for s in sents:
        a = neuron_activations(model, s, DEVICE)
        mid = config.n_layer // 2
        xs.append(a[mid]['x'])
    concept_acts[concept] = np.mean(xs, axis=0)  # (H, N)
    print(f"  {concept}: {len(sents)} sentences")

# Selectivity per neuron
concepts = list(CONCEPT_BANK.keys())
H, N = config.n_head, config.N
neurons = []

for h in range(H):
    for n in range(N):
        acts = {c: float(concept_acts[c][h, n]) for c in concepts}
        best = max(acts, key=lambda c: acts[c])
        best_val = acts[best]
        others = [v for c,v in acts.items() if c != best]
        mean_other = np.mean(others) if others else 0
        sel = (best_val - mean_other) / (best_val + mean_other + 1e-8)

        if best_val > 0.001:
            neurons.append({
                'head':h, 'neuron':n, 'global': h*N+n,
                'concept':best, 'selectivity':round(float(sel),4),
                'activations':{c:round(v,6) for c,v in acts.items()},
            })

neurons.sort(key=lambda x: -x['selectivity'])

# Per-concept fingerprints
fingerprints = {}
for c in concepts:
    cn = [n for n in neurons if n['concept'] == c]
    fingerprints[c] = {
        'count':len(cn), 'top':cn[:20],
        'mean_sel':round(np.mean([n['selectivity'] for n in cn]),4) if cn else 0,
    }

mono_data = {
    'concepts':concepts, 'analysis_layer':config.n_layer//2,
    'total_active':len(neurons), 'total_neurons':H*N,
    'top_200':neurons[:200], 'fingerprints':fingerprints,
}
json.dump(mono_data, open(OUT/"monosemanticity/precomputed.json",'w'))

for c, fp in fingerprints.items():
    print(f"  {c}: {fp['count']} selective neurons, mean_sel={fp['mean_sel']}")
print(" Monosemanticity done")


# %% Cell 9 — Global Sparsity Stats

print("\n" + "="*65)
print(" PHASE 5: Sparsity stats")
print("="*65)

layer_sp = {L: {'x':[], 'y':[]} for L in range(config.n_layer)}
layer_pre = {L: {'frac_neg':[]} for L in range(config.n_layer)}

for text in tqdm(CORPUS, desc="Sparsity"):
    tokens = torch.tensor([list(text.encode('utf-8'))], dtype=torch.long, device=DEVICE)
    with torch.no_grad(), model.extracting() as buf:
        model(tokens)
    for L in range(config.n_layer):
        layer_sp[L]['x'].append(sparsity(buf[L]['x']))
        layer_sp[L]['y'].append(sparsity(buf[L]['y']))
        xp = buf[L]['x_pre'].flatten().numpy()
        layer_pre[L]['frac_neg'].append(float((xp<0).mean()))

all_x = [v for L in layer_sp for v in layer_sp[L]['x']]
all_y = [v for L in layer_sp for v in layer_sp[L]['y']]

stats = {
    'n_sentences': len(CORPUS),
    'per_layer': {str(L): {
        'x_sp':round(np.mean(layer_sp[L]['x']),4),
        'y_sp':round(np.mean(layer_sp[L]['y']),4),
        'pre_relu_neg':round(np.mean(layer_pre[L]['frac_neg']),4),
    } for L in range(config.n_layer)},
    'global': {
        'x_sp':round(np.mean(all_x),4), 'y_sp':round(np.mean(all_y),4),
        'x_active_pct':round((1-np.mean(all_x))*100,2),
        'y_active_pct':round((1-np.mean(all_y))*100,2),
    },
}
json.dump(stats, open(OUT/"sparsity/global_stats.json",'w'), indent=2)
print(f"  x sparsity: {stats['global']['x_sp']:.1%}")
print(f"  y sparsity: {stats['global']['y_sp']:.1%}")
print(" Sparsity done")


# %% Cell 10 — Copy Evolution + Merge Data

print("\n PHASE 6: Evolution & merge")
src = Path("training_telemetry")
if src.exists():
    for f in src.glob("*.json"):
        target_dir = OUT/"merge" if "merge" in f.name else OUT/"evolution"
        shutil.copy2(f, target_dir/f.name)
        print(f"  Copied {f.name}")

# Combined evolution
combined = []
for ef in (OUT/"evolution").glob("evolution_*.json"):
    data = json.load(open(ef))
    lang = ef.stem.replace("evolution_","")
    for e in data: e['lang'] = lang
    combined.extend(data)
combined.sort(key=lambda x: x['iteration'])
json.dump(combined, open(OUT/"evolution/evolution_summary.json",'w'), indent=2)
print(f"  Combined: {len(combined)} points")


# %% Cell 11 — Package

print("\n" + "="*65)
print(" Final package")
print("="*65)

total_size = sum(f.stat().st_size for f in OUT.rglob("*") if f.is_file())
n_files = sum(1 for _ in OUT.rglob("*") if _.is_file())

for sub in sorted(OUT.iterdir()):
    if sub.is_dir():
        n = sum(1 for _ in sub.glob("*"))
        sz = sum(f.stat().st_size for f in sub.glob("*") if f.is_file())
        print(f"  {sub.name}/  ({n} files, {sz/1024:.0f}KB)")
    else:
        print(f"  {sub.name}  ({sub.stat().st_size/1024:.0f}KB)")

print(f"\n  Total: {n_files} files, {total_size/1e6:.1f}MB")

shutil.make_archive("viz_data_complete", 'zip', str(OUT))
print(f"   viz_data_complete.zip ({Path('viz_data_complete.zip').stat().st_size/1e6:.1f}MB)")

try:
    from google.colab import files
    files.download("viz_data_complete.zip")
except ImportError:
    print("  (Download manually)")

print("""
╔═══════════════════════════════════════════════════════════╗
║  DONE!                                                    ║
║                                                           ║
║  telemetry/*.json  → Token Journey (click token, see Eq8) ║
║  graph/*.json      → Force-directed graph visualization   ║
║  synapses/*.json   → Hebbian timeline (Figure 12 style)   ║
║  monosemanticity/* → Concept selectivity dashboard        ║
║  sparsity/*.json   → Aggregate stats + evolution curve    ║
║  hero_tokens/*     → Full 3072-dim vector dumps           ║
╚═══════════════════════════════════════════════════════════╝
""")

Device: cuda
 Loading checkpoints/french/best.pt...
  6L/192D/4H, N=3072/head
  Params: 7,962,624

 PHASE 1: Per-token telemetry


Extracting: 100%|██████████| 15/15 [00:10<00:00,  1.42it/s]


 15 sentences, 1506 hero dumps

  PHASE 2: Graph topology
  Head 0: 94373 edges, 12 clusters, modularity=0.079
  Head 1: 94373 edges, 10 clusters, modularity=0.082
  Head 2: 94373 edges, 10 clusters, modularity=0.080
  Head 3: 94373 edges, 11 clusters, modularity=0.078
 Graph done

 PHASE 3: Synapse tracking
  Tracking 20 synapses from G*[head=0]
  Sentence 0: <F:en>The price was fifty euros and thirty pounds<...


/tmp/ipython-input-3998255527.py:419: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  val = float(torch.dot(rho[s['i']], E_h0[s['j']]))


  Sentence 1: <F:en>The dollar strengthened against the yen toda...
  Sentence 3: <F:en>Germany and France signed the bilateral trea...
  Sentence 6: <F:en>The European Parliament voted on this resolu...
  Sentence 11: <F:en>The budget was discussed. The budget was app...
  Sentence 14: <F:en>The British pound fell after the vote in Lon...
 Synapse tracking done

 PHASE 4: Monosemanticity
  currency: 8 sentences
  country: 8 sentences
  institution: 6 sentences
  action_verb: 6 sentences
  currency: 1875 selective neurons, mean_sel=0.346
  country: 1312 selective neurons, mean_sel=0.3674
  institution: 1813 selective neurons, mean_sel=0.3696
  action_verb: 1959 selective neurons, mean_sel=0.3463
 Monosemanticity done

 PHASE 5: Sparsity stats


Sparsity: 100%|██████████| 15/15 [00:00<00:00, 30.42it/s]


  x sparsity: 94.8%
  y sparsity: 99.0%
 Sparsity done

 PHASE 6: Evolution & merge
  Copied evolution_portuguese.json
  Copied evolution_french.json
  Copied merge_eval.json
  Combined: 36 points

 Final package
  corpus.json  (3KB)
  evolution/  (3 files, 170KB)
  graph/  (4 files, 87KB)
  hero_tokens/  (50 files, 19231KB)
  merge/  (1 files, 1KB)
  meta.json  (0KB)
  monosemanticity/  (1 files, 52KB)
  sparsity/  (1 files, 1KB)
  synapses/  (1 files, 113KB)
  telemetry/  (15 files, 33699KB)

  Total: 78 files, 54.6MB
   viz_data_complete.zip (19.6MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


╔═══════════════════════════════════════════════════════════╗
║  DONE!                                                    ║
║                                                           ║
║  telemetry/*.json  → Token Journey (click token, see Eq8) ║
║  graph/*.json      → Force-directed graph visualization   ║
║  synapses/*.json   → Hebbian timeline (Figure 12 style)   ║
║  monosemanticity/* → Concept selectivity dashboard        ║
║  sparsity/*.json   → Aggregate stats + evolution curve    ║
║  hero_tokens/*     → Full 3072-dim vector dumps           ║
╚═══════════════════════════════════════════════════════════╝

